In [ ]:
# ML Model 5 — Bed Demand Forecasting

## Objective
Predict future hospital bed demand.

## Problem Type
Time-Series Forecasting

## Target Variable
occupied_beds

## Evaluation Metrics
- MAE
- RMSE

In [1]:
# Model 5 — Bed Demand Forecasting
# This model uses historical daily occupied-bed counts
# to forecast future hospital bed demand.

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ==========================================================
# 1. LOAD BED DATA
# ==========================================================

file_path = "../data/processed/beds/beds_clean.csv"

df = pd.read_csv(file_path)

df["record_date"] = pd.to_datetime(df["record_date"])

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ==========================================================
# 2. AGGREGATE DAILY BED DEMAND
# ==========================================================
# We want total occupied beds across the hospital each day.

daily = (
    df.groupby("record_date")["occupied_beds"]
      .sum()
      .reset_index()
      .sort_values("record_date")
)

print("\nDaily bed-demand dataset:")
print(daily.head())

print("\nNumber of days:", len(daily))


# ==========================================================
# 3. CREATE TIME FEATURES
# ==========================================================

daily["dayofweek"] = daily["record_date"].dt.dayofweek
daily["month"] = daily["record_date"].dt.month
daily["dayofyear"] = daily["record_date"].dt.dayofyear

daily["time_index"] = np.arange(len(daily))


# ==========================================================
# 4. CREATE LAG FEATURES
# ==========================================================
# Previous day, previous week and previous month.

daily["lag_1"] = daily["occupied_beds"].shift(1)
daily["lag_7"] = daily["occupied_beds"].shift(7)
daily["lag_30"] = daily["occupied_beds"].shift(30)


# Remove rows where lag values are unavailable

daily = daily.dropna().reset_index(drop=True)


# ==========================================================
# 5. DEFINE FEATURES AND TARGET
# ==========================================================

features = [
    "time_index",
    "dayofweek",
    "month",
    "dayofyear",
    "lag_1",
    "lag_7",
    "lag_30"
]

X = daily[features]

y = daily["occupied_beds"]


# ==========================================================
# 6. TIME-BASED TRAIN / TEST SPLIT
# ==========================================================
# No random shuffle because this is time-series data.

split = int(len(daily) * 0.80)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


# ==========================================================
# 7. CREATE MODEL
# ==========================================================

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)


# ==========================================================
# 8. TRAIN MODEL
# ==========================================================

print("\nTraining Bed Demand model...")

model.fit(X_train, y_train)

print("Training completed successfully!")


# ==========================================================
# 9. GENERATE PREDICTIONS
# ==========================================================

print("\nGenerating predictions...")

y_pred = model.predict(X_test)

print("Predictions generated successfully!")


# ==========================================================
# 10. EVALUATE MODEL
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))


print("\n========== BED DEMAND MODEL PERFORMANCE ==========")

print(f"MAE  : {mae:.2f} beds")
print(f"RMSE : {rmse:.2f} beds")


# ==========================================================
# 11. SAMPLE RESULTS
# ==========================================================

results = pd.DataFrame({
    "Date": daily.iloc[split:]["record_date"].values,
    "Actual": y_test.values,
    "Predicted": np.round(y_pred, 0)
})

print("\n========== SAMPLE FORECAST RESULTS ==========")

print(results.head(10).to_string(index=False))

Dataset shape: (15344, 8)

Columns:
['bed_record_id', 'department_id', 'record_date', 'bed_type', 'total_beds', 'occupied_beds', 'available_beds', 'occupancy_rate']

Daily bed-demand dataset:
  record_date  occupied_beds
0  2023-01-01            239
1  2023-01-02            231
2  2023-01-03            230
3  2023-01-04            243
4  2023-01-05            225

Number of days: 1096

Training data: (852, 7)
Testing data: (214, 7)

Training Bed Demand model...
Training completed successfully!

Generating predictions...
Predictions generated successfully!

========== BED DEMAND MODEL PERFORMANCE ==========
MAE  : 5.28 beds
RMSE : 6.59 beds

========== SAMPLE FORECAST RESULTS ==========
      Date  Actual  Predicted
2025-06-01     206      204.0
2025-06-02     194      202.0
2025-06-03     203      198.0
2025-06-04     206      203.0
2025-06-05     194      202.0
2025-06-06     195      197.0
2025-06-07     198      199.0
2025-06-08     192      197.0
2025-06-09     194      194.0
2025-

In [2]:
# Display only the Bed Demand model metrics.

print("========== BED DEMAND MODEL PERFORMANCE ==========")
print(f"MAE  : {mae:.2f} beds")
print(f"RMSE : {rmse:.2f} beds")

========== BED DEMAND MODEL PERFORMANCE ==========
MAE  : 5.28 beds
RMSE : 6.59 beds


In [3]:
# Save the trained Bed Demand Forecasting model.

import joblib

model_path = "../ml/models/bed_demand_forecasting_model.pkl"

joblib.dump(model, model_path)

print("Bed Demand Forecasting model saved successfully!")
print("Saved at:", model_path)

Bed Demand Forecasting model saved successfully!
Saved at: ../ml/models/bed_demand_forecasting_model.pkl
